In [31]:
import tkinter as tk
from tkinter import ttk, messagebox, simpledialog

# Buat Class bernama Vending Machine yang berisi fungsi - fungsi
class VendingMachine:
# Fungsi Membaca data dari file dengan objek self
    def load_data(self):
        tickets = []
        try:
            # buka file dalam mode baca
            with open ("tickets.txt" , "r") as file:
               for line in file:
                   if ":" in line:
                    kota, harga = line.strip().split(":")
                    kota = kota.strip()
                    harga = int(harga.strip().replace("Rp.", "").replace(" ", ""))
                    tickets.append({"kota": kota, "harga": harga})
        #jika file tidak ditemukan maka lewat               
        except FileNotFoundError:
            pass
        return tickets
# Fungsi menyimpan data dari file
    def save_data (self,tickets):
         # buka file dalam mode tulis (overwrite)
        with open ("tickets.txt" , "w") as file:
            for ticket in tickets:
                file.write(f"{ticket['kota']} : Rp. {ticket['harga']}\n")
                
# Class yang berisi GUI dengan fungsinya    
class VendingGUI:
    # fungsi untuk membuat tampilan kosong
    def __init__(self, root):
        self.root = root
        self.root.title("Railway Ticket Vending System")
        self.root.geometry ("500x400")
        self.root.configure(bg="#f0f0f0")

        self.vm = VendingMachine()
        self.tickets = self.vm.load_data()   
        
        title = tk.Label(root, text=" Railway Ticket Vending System",
                         font=("Times New Roman", 16, "bold"), bg="#f0f0f0")
        title.pack(pady=10)

        self.tree = ttk.Treeview(root, columns=("Kota", "Harga"), show="headings")
        self.tree.heading("Kota", text="Kota Tujuan")
        self.tree.heading("Harga", text="Harga Tiket")

        self.tree.pack(pady=10, fill="both", expand=True)

        self.refresh_list()

        frame_btn = tk.Frame(root, bg="#f0f0f0")
        frame_btn.pack(pady=10)

        tk.Button(frame_btn, text="Tambah", width=12, bg="#4CAF50", fg="white",
                  command=self.tambah_kota).grid(row=0, column=0, padx=5)

        tk.Button(frame_btn, text="Hapus", width=12, bg="#f44336", fg="white",
                  command=self.hapus_tiket).grid(row=0, column=1, padx=5)

        tk.Button(frame_btn, text="Beli", width=12, bg="#2196F3", fg="white",
                  command=self.beli_tiket).grid(row=0, column=2, padx=5)

        tk.Button(frame_btn, text="Keluar", width=12, bg="gray", fg="white",
                  command=self.keluar).grid(row=0, column=3, padx=5)

    # Fungsi untuk merefresh data setiap selesai dijalankan
    def refresh_list(self):
        for row in self.tree.get_children():
            self.tree.delete(row)

        for t in self.tickets:
            self.tree.insert("", "end", values=(t["kota"], f"Rp {t['harga']}"))


    # Fungsi untuk menambahkan kota dan sekaligus menampilkan di layarnya
    def tambah_kota(self):
        window = tk.Toplevel(self.root)
        window.title("Tambah Tiket")
        
        tk.Label(window, text = "Kota").pack()
        kota_entry = tk.Entry(window)
        kota_entry.pack()

        tk.Label(window, text = "Harga").pack()
        harga_entry = tk.Entry(window)
        harga_entry.pack()

        def simpan():
            try:
                kota = kota_entry.get()
                harga = int(harga_entry.get())

                self.tickets.append({"kota": kota, "harga": harga})
                self.vm.save_data(self.tickets)
                self.refresh_list()

                messagebox.showinfo("Sukses", "Data Telah Ditambahkan")
                window.destroy()
            except:
                messagebox.showerror ("Gagal", "Input Salah!")
        tk.Button(window, text= "Simpan", command=simpan).pack(pady=10)

    # Fungsi untuk membeli tiket dengan memilih tujuan     
    def beli_tiket(self):
        try:
            selected = self.tree.selection()
            index = self.tree.index(selected[0])
        except:
            messagebox.showerror("Gagal", "Pilih Tiket Dahulu")
            return

        window = tk.Toplevel(self.root)
        window.title("Beli Tiket")

        tk.Label(window, text = "Jumlah Tiket").pack()
        jumlah_entry = tk.Entry(window)
        jumlah_entry.pack()

        def proses():
            try:
                jumlah = int(jumlah_entry.get())
                data = self.tickets[index]

                total = data["harga"] * jumlah
    
                messagebox.showinfo(
                    "===Struk Pembelian===",
                    f"Tujuan : {data['kota']}\n"
                    f"Harga  : Rp {data['harga']}\n"
                    f"Jumlah : {jumlah}\n"
                    f"Total  : Rp {total}"
                )
    
                window.destroy()
            except:
                messagebox.showerror("Error", "Input jumlah tidak valid!")

        tk.Button(window, text="Beli", command=proses).pack(pady=10) 

    # fungsi untuk menghapus tiket
    def hapus_tiket(self):
        try:
            selected = self.tree.selection()
            index = self.tree.index(selected[0])

            self.tickets.pop(index)
            self.vm.save_data(self.tickets)
            self.refresh_list()

            messagebox.showinfo("Sukses", "Data Telah Dihapus")
        except:
            messagebox.showerror("Gagal", "Pilih Tiket Dahulu")


    # fungsi untuk keluar dari menu
    def keluar (self):
        konfirm = messagebox.askyesno ("Konfirmasi", "Yakin Ingin Keluar?")
        if konfirm:
            self.root.destroy()
            
if __name__ == "__main__":
    
        root = tk.Tk()
        app = VendingGUI(root)
        root.mainloop()